# Step 1:  Load the student table and the teacher’s proxies

In [1]:
from google.colab import drive
drive.mount('/content/drive')

%run /content/drive/MyDrive/m5_project/notebooks/features.ipynb      # student table + hierarchy + scorer

import lightgbm as lgb, gc

# free memory: tables this notebook does not need (skips any that don't exist)
for name in ['df', 'price_feats', 'teacher', 'test', 'p']:
    globals().pop(name, None)
gc.collect()

# teacher proxies (saved in teacher.ipynb)
OUT   = DRIVE / 'outputs'
saved = pd.read_parquet(OUT / 'teacher_proxies.parquet')
saved.columns = saved.columns.astype(int)                            # '1914' -> 1914
proxies = {name: saved.loc[name].reindex(sales_by_level[name].index) for name in TEACHER_LEVELS}

print({name: table.shape for name, table in proxies.items()})
print('student:', student.shape)

Mounted at /content/drive
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
(6096944, 8)
days: 847 -> 1941
L1_total            1 series
L2_store            2 series
L3_cat              3 series
L4_dept             7 series
L5_store_cat        6 series
L6_store_dept      14 series
L7_item          3049 series
L8_item_store    6098 series
(6098, 1095)
L1_total            1 series x 1095 days
L2_store            2 series x 1095 days
L3_cat              3 series x 1095 days
L4_dept             7 series x 1095 days
L5_store_cat        6 series x 1095 days
L6_store_dept      14 series x 1095 days
L7_item          3049 series x 1095 days
L8_item_store    6098 series x 1095 days
Level 1: Sum of both the stores
d       847   848   849   850   851
Total  7745  8747  7996  8197  6266
--------

# Step 2: Student features, forecast windows and clusters

In [2]:
# clues the student uses (same for every trial -> fair comparison)
ID_COLS    = ['item_id', 'dept_id', 'cat_id', 'store_id']
S_FEATURES = ([f'lag_{l}' for l in LAGS]                        # past sales, same weekday
              + [f'roll_mean_{w}' for w in WINDOWS]             # usual level
              + ['sell_price', 'price_vs_usual']                # price clues
              + ['wday', 'month', 'day_of_month', 'event_type', 'snap_CA']   # calendar
              + ID_COLS)                                        # which product / dept / store

for col in ID_COLS:
    student[col] = student[col].astype('category')

# the two forecast windows
VAL_DAYS  = list(range(VAL_START, TEST_START))
TEST_DAYS = list(range(TEST_START, TEST_START + HORIZON))

CLUSTER_SETS = {'store': 'store_id', 'dept': 'dept_id'}

for set_name, col in CLUSTER_SETS.items():
    sizes = student[col].value_counts().sort_index()
    print(f'{set_name} clusters ({len(sizes)}):', sizes.to_dict())

print('features:', len(S_FEATURES))

store clusters (2): {'CA_1': 2946957, 'CA_2': 2759631}
dept clusters (7): {'FOODS_1': 429335, 'FOODS_2': 595096, 'FOODS_3': 1562719, 'HOBBIES_1': 820141, 'HOBBIES_2': 286209, 'HOUSEHOLD_1': 1011906, 'HOUSEHOLD_2': 1001182}
features: 20


# Step 3: One trial = one model per cluster, two jobs

In [3]:
JOBS = {   # name: (last training day, days to forecast)
    'val':  (TRAIN_END,      VAL_DAYS),     # Job A -> graded by TCV
    'test': (TEST_START - 1, TEST_DAYS),    # Job B -> graded by H-Pro (proxies) and Gold (real test)
}

def to_grid(rows, preds, days):
    """Long predictions -> bottom grid (6,098 series x 28 days); not-on-sale days = 0."""
    grid = rows[['series_id', 'd']].assign(pred=preds).pivot(index='series_id', columns='d', values='pred')
    return grid.reindex(index=bottom.index, columns=days).fillna(0)

def run_trial(params, set_name, seed):
    """Train one model per cluster for both jobs; return {'val': grid, 'test': grid}."""
    col = CLUSTER_SETS[set_name]
    grids = {}
    for job, (last_day, days) in JOBS.items():
        parts = []
        for cluster in student[col].cat.categories:
            in_cluster = student[col] == cluster
            train_rows = student[in_cluster & (student.d <= last_day)]
            pred_rows  = student[in_cluster & student.d.isin(days)]
            model = lgb.LGBMRegressor(objective='tweedie', random_state=seed, n_jobs=-1, verbose=-1, **params)
            model.fit(train_rows[S_FEATURES], train_rows['sales'])
            parts.append(to_grid(pred_rows, model.predict(pred_rows[S_FEATURES]), days))
        grids[job] = sum(parts)          # clusters don't overlap, so adding = joining
    return grids


_g = run_trial(dict(n_estimators=10), 'store', seed=0)
print({job: g.shape for job, g in _g.items()})

{'val': (6098, 28), 'test': (6098, 28)}


# Step 4: Random search over student settings (with checkpoints)
- 12 random LightGBM settings (seed 0). Each trial is trained in every store cluster and every dept cluster,
for both jobs (validation and test)

In [4]:
S_SPACE = {
    'learning_rate':          [0.03, 0.05, 0.1, 0.2],
    'num_leaves':             [31, 63, 127, 255],
    'min_child_samples':      [20, 50, 100, 200],
    'n_estimators':           [200, 400, 700],
    'colsample_bytree':       [0.6, 0.8, 1.0],
    'tweedie_variance_power': [1.1, 1.3, 1.5],
}
N_TRIALS = 12
SEED     = 0
rng = np.random.default_rng(SEED)
TRIAL_PARAMS = [{k: rng.choice(v).item() for k, v in S_SPACE.items()} for _ in range(N_TRIALS)]

CKPT = OUT / f'student_trials_seed{SEED}'
CKPT.mkdir(exist_ok=True)

import time
RUN_TRIALS = range(0, N_TRIALS)          # teammates can split: range(0, 6) and range(6, 12)
for t in RUN_TRIALS:
    params = TRIAL_PARAMS[t]
    for set_name in CLUSTER_SETS:
        f_val, f_test = CKPT / f'{set_name}_{t:02d}_val.parquet', CKPT / f'{set_name}_{t:02d}_test.parquet'
        if f_val.exists() and f_test.exists():
            print(f'trial {t:02d} {set_name:<5} already done, skipping'); continue
        start = time.time()
        grids = run_trial(params, set_name, SEED)
        for job, f in [('val', f_val), ('test', f_test)]:
            g = grids[job].copy(); g.columns = g.columns.astype(str); g.to_parquet(f)
        print(f'trial {t:02d} {set_name:<5} done in {(time.time() - start) / 60:.1f} min')

trial 00 store already done, skipping
trial 00 dept  already done, skipping
trial 01 store already done, skipping
trial 01 dept  already done, skipping
trial 02 store already done, skipping
trial 02 dept  already done, skipping
trial 03 store already done, skipping
trial 03 dept  already done, skipping
trial 04 store already done, skipping
trial 04 dept  already done, skipping
trial 05 store already done, skipping
trial 05 dept  already done, skipping
trial 06 store already done, skipping
trial 06 dept  already done, skipping
trial 07 store already done, skipping
trial 07 dept  already done, skipping
trial 08 store already done, skipping
trial 08 dept  already done, skipping
trial 09 store already done, skipping
trial 09 dept  already done, skipping
trial 10 store already done, skipping
trial 10 dept  already done, skipping
trial 11 store already done, skipping
trial 11 dept  already done, skipping
